### Iteration Group

In [1]:
import copy
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from HybridCloud import *
import matplotlib.pyplot as plt

# -----------------------------
# Helpers: per-job DF + summary
# -----------------------------

def build_job_energy_df(job_records: dict) -> pd.DataFrame:
    rows = []
    for job_id, rec in job_records.items():
        e_qpu = float(rec.get("energy_qpu_kwh", 0.0) or 0.0)
        e_cpu = float(rec.get("energy_cpu_kwh", 0.0) or 0.0)
        e_tot = float(rec.get("energy_total_kwh", e_qpu + e_cpu) or (e_qpu + e_cpu))
        c_tot = float(rec.get("cost_energy_total", 0.0) or 0.0)

        # Avoid divide-by-zero
        if e_tot > 0:
            phi_qpu = e_qpu / e_tot
            phi_cpu = e_cpu / e_tot
        else:
            phi_qpu = 0.0
            phi_cpu = 0.0

        # ---- NEW: wait/turn/makespan from your record structure ----
        qpu_wait_list = rec.get("qpu_wait", []) or []
        cpu_wait_list = rec.get("cpu_wait", []) or []
        qpu_turn_list = rec.get("qpu_turn", []) or []
        cpu_turn_list = rec.get("cpu_turn", []) or []
        makespan_list = rec.get("makespan", []) or []

        qpu_wait_s = float(np.sum(qpu_wait_list)) if isinstance(qpu_wait_list, list) else float(qpu_wait_list or 0.0)
        cpu_wait_s = float(np.sum(cpu_wait_list)) if isinstance(cpu_wait_list, list) else float(cpu_wait_list or 0.0)

        # "turn" here is per-segment turnaround; sum gives total turnaround across all segments
        qpu_turn_s = float(np.sum(qpu_turn_list)) if isinstance(qpu_turn_list, list) else float(qpu_turn_list or 0.0)
        cpu_turn_s = float(np.sum(cpu_turn_list)) if isinstance(cpu_turn_list, list) else float(cpu_turn_list or 0.0)

        # Job-level end-to-end turnaround time (your record calls it makespan)
        job_turnaround_s = float(makespan_list[0]) if isinstance(makespan_list, list) and len(makespan_list) > 0 else float(rec.get("makespan", 0.0) or 0.0)

        rows.append({
            "job_id": job_id,
            "energy_qpu_kwh": e_qpu,
            "energy_cpu_kwh": e_cpu,
            "energy_total_kwh": e_tot,
            "cost_energy_total": c_tot,
            "phi_qpu": phi_qpu,
            "phi_cpu": phi_cpu,
            "qpu_time_s": float(rec.get("qpu_time_s", 0.0) or 0.0),
            "cpu_time_s": float(rec.get("cpu_time_s", 0.0) or 0.0),

            # ---- NEW columns ----
            "qpu_wait_s": qpu_wait_s,
            "cpu_wait_s": cpu_wait_s,
            "wait_total_s": qpu_wait_s + cpu_wait_s,
            "qpu_turn_s": qpu_turn_s,
            "cpu_turn_s": cpu_turn_s,
            "turn_total_s": qpu_turn_s + cpu_turn_s,
            "job_turnaround_s": job_turnaround_s,   # end-to-end
        })

    df = pd.DataFrame(rows).sort_values("job_id").reset_index(drop=True)
    return df

def get_summary(df: pd.DataFrame, PRINT_DATA) -> dict:
    def p95(x):
        return float(np.percentile(x, 95))

    summary = {
        "number_of_jobs": int(len(df)),

        # ---- Mean energy ----
        "mean_energy_qpu_kwh": float(df["energy_qpu_kwh"].mean()),
        "mean_energy_cpu_kwh": float(df["energy_cpu_kwh"].mean()),
        "mean_energy_total_kwh": float(df["energy_total_kwh"].mean()),

        # ---- Std deviation (energy variability) ----
        "std_energy_qpu_kwh": float(df["energy_qpu_kwh"].std()),
        "std_energy_cpu_kwh": float(df["energy_cpu_kwh"].std()),
        "std_energy_total_kwh": float(df["energy_total_kwh"].std()),

        # ---- Cost statistics ----
        "mean_cost_per_job": float(df["cost_energy_total"].mean()),
        "std_cost_per_job": float(df["cost_energy_total"].std()),        
        "median_cost_per_job": float(df["cost_energy_total"].median()),
        "p95_cost_per_job": p95(df["cost_energy_total"]),

        # ---- Energy fractions ----
        "mean_phi_qpu": float(df["phi_qpu"].mean()),
        "mean_phi_cpu": float(df["phi_cpu"].mean()),
        "std_phi_qpu": float(df["phi_qpu"].std()),
        "std_phi_cpu": float(df["phi_cpu"].std()),        

        # ---- Time statistics ----
        "mean_qpu_time_s": float(df["qpu_time_s"].mean()),
        "mean_cpu_time_s": float(df["cpu_time_s"].mean()),
        "std_qpu_time_s": float(df["qpu_time_s"].std()),
        "std_cpu_time_s": float(df["cpu_time_s"].std()),        
        "p95_qpu_time_s": p95(df["qpu_time_s"]),
        "p95_cpu_time_s": p95(df["cpu_time_s"]),
    }

    # ---- NEW: wait + turnaround summaries (only if columns exist) ----
    if "qpu_wait_s" in df.columns:
        summary.update({
            "mean_qpu_wait_s": float(df["qpu_wait_s"].mean()),
            "mean_cpu_wait_s": float(df["cpu_wait_s"].mean()),
            "mean_wait_total_s": float(df["wait_total_s"].mean()),
            "std_qpu_wait_s": float(df["qpu_wait_s"].std()),
            "std_cpu_wait_s": float(df["cpu_wait_s"].std()),
            "std_wait_total_s": float(df["wait_total_s"].std()),
            "p95_wait_total_s": p95(df["wait_total_s"]),
        })

    # Job-level end-to-end turnaround (your record's 'makespan')
    if "job_turnaround_s" in df.columns:
        summary.update({
            "mean_job_turnaround_s": float(df["job_turnaround_s"].mean()),
            "std_job_turnaround_s": float(df["job_turnaround_s"].std()),
            "median_job_turnaround_s": float(df["job_turnaround_s"].median()),
            "p95_job_turnaround_s": p95(df["job_turnaround_s"]),
        })

    # Segment-level turnaround sums (optional but nice for diagnosing pipeline overhead)
    if "turn_total_s" in df.columns:
        summary.update({
            "mean_turn_total_s": float(df["turn_total_s"].mean()),
            "std_turn_total_s": float(df["turn_total_s"].std()),
            "p95_turn_total_s": p95(df["turn_total_s"]),
        })

    # Optional: fraction of end-to-end time spent waiting (queueing)
    if "wait_total_s" in df.columns and "job_turnaround_s" in df.columns:
        denom = df["job_turnaround_s"].replace(0, np.nan)
        wait_frac = (df["wait_total_s"] / denom).fillna(0.0)
        summary.update({
            "mean_wait_fraction": float(wait_frac.mean()),
            "std_wait_fraction": float(wait_frac.std()),
            "p95_wait_fraction": p95(wait_frac),
        })
    if PRINT_DATA:
        print("\n=== Energy/Cost Summary ===")
        for k, v in summary.items():
            if isinstance(v, float):
                print(f"{k:>22}: {v:.6f}")
            else:
                print(f"{k:>22}: {v}")
    return summary

def extract_iterations_from_filename(path: str) -> int:
    """
    Extracts k from filenames like:
      5000-job_iter_3.csv
      synth_job_batches/5000-job_iter_12.csv
    """
    m = re.search(r"iter_(\d+)", str(path))
    if not m:
        raise ValueError(f"Could not extract iterations from filename: {path}")
    return int(m.group(1))


PRINTLOG = False

# -----------------------------
# 0) Devices (create fresh per run recommended)
# -----------------------------
def make_devices(printlog=PRINTLOG):
    # QPUs
    ibm_strasbourg = IBM_Strasbourg(env=None, name="QPU-1", printlog=printlog)
    ibm_brussels   = IBM_Brussels(env=None, name="QPU-2", printlog=printlog)
    # CPUs
    ryzen1 = AMDRyzen("CPU-1", env=None)
    ryzen2 = AMDRyzen("CPU-2", env=None)
    return [ibm_strasbourg, ibm_brussels], [ryzen1, ryzen2]


# -----------------------------
# 1) Base cost_config (fixed for this experiment)
# -----------------------------
base_cost_config = {
  "energy": {
    "electricity_price_per_kwh": 0.18,
    "default_qpu_power_kw": 50.0,
    "qpu_power_kw": {"QPU-1": 70.0, "QPU-2": 60.0},
    "cpu_power_kw": {"CPU-1": 5.0, "CPU-2": 6.5},
    "cpu_power_model": "affine",
    "default_cpu_idle_kw": 0.22,
    "default_cpu_peak_kw": 0.75,
    "default_cpu_capacity_units": 16,
    "debug_energy": False
  }
}


# -----------------------------
# 2) Create env for a run
# -----------------------------
def make_env(file_path: str, cost_config: dict, printlog=PRINTLOG):
    qpus, cpus = make_devices(printlog=printlog)
    sim_env = HybridCloudSimEnv(
        qpu_devices=qpus,
        cpu_devices=cpus,
        broker_class=HybridBroker,
        job_feed_method='dispatcher',
        file_path=file_path,
        job_generation_model=None,
        printlog=printlog,
        cost_config=cost_config
    )
    return sim_env


# -----------------------------
# 3) Iteration-group runner
# -----------------------------
def run_iteration_groups(job_csv_list, *, save_per_job_csv=True, out_dir="runs", PRINT_DATA) -> pd.DataFrame:
    """
    Runs one simulation per CSV (each CSV corresponds to a fixed req_iterations group).
    Returns a summary DataFrame with one row per iteration group.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for csv_path in job_csv_list:
        k = extract_iterations_from_filename(csv_path)

        cost_config = copy.deepcopy(base_cost_config)  # keep identical across groups
        
        print(f"\n=== Running iteration group: k={k} | file={csv_path} ===")
        
        sim_env = make_env(file_path=csv_path, cost_config=cost_config, printlog=PRINT_DATA)

        # Optional sanity prints
        energy_cfg = sim_env.cost_config.get("energy", {})

        if PRINT_DATA:
            print(f"electricity_price_per_kwh: {energy_cfg.get('electricity_price_per_kwh')}")
            print(f"cpu_power_model: {energy_cfg.get('cpu_power_model')}")
            print(f"default_cpu_idle_kw: {energy_cfg.get('default_cpu_idle_kw')}")

        sim_env.run()

        job_records = sim_env.job_records_manager.job_records
        df_jobs = build_job_energy_df(job_records)

        # Save per-job records for plotting p50/p95 curves later
        if save_per_job_csv:
            per_job_path = out_dir / f"results_iter_{k}_jobs.csv"
            df_jobs.to_csv(per_job_path, index=False)
            
        if PRINT_DATA:
            print(f"Saved per-job records: {per_job_path} ({len(df_jobs)} jobs)")

        summary = get_summary(df_jobs, PRINT_DATA)
        summary["iterations"] = k
        summary["workload_csv"] = str(csv_path)
        rows.append(summary)

    df_summary = pd.DataFrame(rows).sort_values("iterations").reset_index(drop=True)
    return df_summary


# -----------------------------
# 4) Execute iteration sweep
# -----------------------------
JOB_CSV = [
    "synth_job_batches/iter-job-batches/3000-job_iter_3.csv",
    "synth_job_batches/iter-job-batches/3000-job_iter_6.csv",
    "synth_job_batches/iter-job-batches/3000-job_iter_9.csv",
    "synth_job_batches/iter-job-batches/3000-job_iter_12.csv",
    "synth_job_batches/iter-job-batches/3000-job_iter_15.csv",
    "synth_job_batches/iter-job-batches/3000-job_iter_18.csv", 
    "synth_job_batches/iter-job-batches/3000-job_iter_21.csv"
]

df_sweep = run_iteration_groups(JOB_CSV, save_per_job_csv=True, out_dir="runs", PRINT_DATA=False)
df_sweep.to_csv("synth_job_batches/iteration_sweep_summary-21.csv", index=False)
print("\nSaved iteration_sweep_summary-21.csv")

# print(df_sweep)



=== Running iteration group: k=3 | file=synth_job_batches/iter-job-batches/3000-job_iter_3.csv ===
2 QPU(s), 2 CPU(s)
0.00: SIMULATION STARTED
0 has been processed.
500 has been processed.
1000 has been processed.
1500 has been processed.
2000 has been processed.
2500 has been processed.
2887.56: SIMULATION ENDED
Number of jobs processed: 3000

                 FINAL QUANTUM CLOUD SUMMARY                
Total Operational Lifetime : 2887.56 seconds
Global Cumulative QPU Util : 15.74%
Global Cumulative CPU Util : 1.84%
Global Cumulative Mem Util : 2.56%
------------------------------------------------------------
Individual Device Endstates:


=== Running iteration group: k=6 | file=synth_job_batches/iter-job-batches/3000-job_iter_6.csv ===
2 QPU(s), 2 CPU(s)
0.00: SIMULATION STARTED
0 has been processed.
500 has been processed.
1000 has been processed.
1500 has been processed.
2000 has been processed.
2500 has been processed.
2892.08: SIMULATION ENDED
Number of jobs processed: 3000

 